# LangChain Run Log Stream Reference

# `LogEntry: TypedDict`

Represents one sub-run recorded inside a streamed run log.

```python
id: str # ID of the sub-run
name: str # Name of the object being run
type: str # Run type, such as prompt, chain, or LLM
tags: list[str] # Tags associated with the run
metadata: dict[str, Any] # Metadata associated with the run
start_time: str # ISO-8601 timestamp indicating when the run started
streamed_output_str: list[str] # LLM tokens streamed by the run when applicable
streamed_output: list[Any] # Output chunks streamed by the run when available
inputs: NotRequired[Any | None] # Run inputs when available
final_output: Any | None # Final output after successful completion
end_time: str | None # ISO-8601 completion timestamp when available
```

`inputs` is optional and is not currently available through `astream_log`.

In [ ]:
from langchain_core.tracers.log_stream import LogEntry # Import the LogEntry type


log_entry: LogEntry = { # Create one sub-run log entry
    "id": "run-101", # Store the sub-run ID
    "name": "double", # Store the runnable name
    "type": "chain", # Store the run type
    "tags": ["math", "demo"], # Store tags associated with the run
    "metadata": {"source": "jupyter"}, # Store additional metadata
    "start_time": "2026-07-14T10:00:00+00:00", # Store the ISO-8601 start time
    "streamed_output_str": [], # Store streamed LLM tokens when applicable
    "streamed_output": [10], # Store streamed output chunks
    "inputs": {"number": 5}, # Store the optional run input
    "final_output": 10, # Store the final output
    "end_time": "2026-07-14T10:00:01+00:00", # Store the completion time
} # Finish creating the log entry

print("Run ID:", log_entry["id"]) # Display the sub-run ID
print("Run name:", log_entry["name"]) # Display the sub-run name
print("Run type:", log_entry["type"]) # Display the run type
print("Tags:", log_entry["tags"]) # Display the tags
print("Metadata:", log_entry["metadata"]) # Display the metadata
print("Inputs:", log_entry.get("inputs")) # Safely access the optional inputs field
print("Streamed output:", log_entry["streamed_output"]) # Display streamed chunks
print("Final output:", log_entry["final_output"]) # Display the final output
print("Start time:", log_entry["start_time"]) # Display the start time
print("End time:", log_entry["end_time"]) # Display the end time

# `RunState: TypedDict`

Represents the reconstructed state of a streamed root run.

```python
id: str # ID of the root run
streamed_output: list[Any] # Output chunks streamed by Runnable.stream()
final_output: Any | None # Aggregated run output as it becomes available
name: str # Name of the object being run
type: str # Run type, such as prompt, chain, or LLM
logs: dict[str, LogEntry] # Included sub-runs keyed by generated log name
```

When filters are configured, `logs` contains only matching sub-runs.


In [ ]:
from langchain_core.tracers.log_stream import LogEntry, RunState # Import the run-state types


child_log: LogEntry = { # Create information for one child run
    "id": "child-run-1", # Store the child run ID
    "name": "double", # Store the child run name
    "type": "chain", # Store the child run type
    "tags": ["math"], # Store the child run tags
    "metadata": {"source": "jupyter"}, # Store additional metadata
    "start_time": "2026-07-14T10:00:00+00:00", # Store the start time
    "streamed_output_str": [], # Store streamed text tokens
    "streamed_output": [10], # Store streamed output chunks
    "inputs": {"number": 5}, # Store the child run input
    "final_output": 10, # Store the child run output
    "end_time": "2026-07-14T10:00:01+00:00", # Store the completion time
}


run_state: RunState = { # Create the reconstructed root-run state
    "id": "root-run-1", # Store the root run ID
    "streamed_output": ["Final value: 10"], # Store output chunks from the root runnable
    "final_output": "Final value: 10", # Store the aggregated final output
    "name": "number_pipeline", # Store the root run name
    "type": "chain", # Store the root run type
    "logs": { # Store included child runs
        "double": child_log, # Add the child log using its generated log name
    },
}


print("Root run ID:", run_state["id"]) # Display the root run ID
print("Root run name:", run_state["name"]) # Display the root run name
print("Final output:", run_state["final_output"]) # Display the final output
print("Streamed output:", run_state["streamed_output"]) # Display streamed chunks

print("\nChild runs:") # Display a heading

for log_name, log_entry in run_state["logs"].items(): # Visit every included child run
    print("Log name:", log_name) # Display the generated log name
    print("Run type:", log_entry["type"]) # Display the child run type
    print("Inputs:", log_entry.get("inputs")) # Display the child inputs
    print("Output:", log_entry["final_output"]) # Display the child output

# `RunLogPatch`

Represents JSON Patch operations that can reconstruct a run state from an empty value.

## Fields

```python
ops: list[dict[str, Any]] # Ordered JSON Patch operations
```

## Constructor

```python
RunLogPatch(
    *ops: dict[str, Any], # JSON Patch operations stored in order
) -> None
```

## Methods

### `__add__`

Combines this patch with another exact `RunLogPatch` instance and applies all operations to an empty state.

```python
__add__(
    self,
    other: RunLogPatch | Any, # Patch to append
) -> RunLog # Reconstructed run log containing the combined operations
```

The operations from `other` are appended after this patch's operations. A deep copy of the combined operations is applied with `jsonpatch.apply_patch()`.

Raises `TypeError` unless `type(other) is RunLogPatch`.

### `__repr__`

Returns a formatted representation of the stored operations.

```python
@override
__repr__(
    self,
) -> str
```

### `__eq__`

Compares two patches by their operation lists.

```python
@override
__eq__(
    self,
    other: object, # Object to compare
) -> bool # Whether both objects are patches with equal operations
```

## Behaviour

`RunLogPatch` instances are unhashable.

# `RunLog: RunLogPatch`

Represents the current run state together with every JSON Patch operation used to produce it.

## Fields

```python
state: RunState # Current state produced by applying the operations in sequence
```

It inherits `ops` from `RunLogPatch`.

## Constructor

```python
RunLog(
    *ops: dict[str, Any], # JSON Patch operations associated with the state
    state: RunState, # Current run-log state
) -> None
```

## Methods

### `__add__`

Applies another exact `RunLogPatch` to the current state.

```python
__add__(
    self,
    other: RunLogPatch | Any, # Patch to apply
) -> RunLog # New run log containing the updated state and combined operations
```

The new state is produced by applying `other.ops` to `state`, and the returned log stores both operation sequences.

Raises `TypeError` unless `type(other) is RunLogPatch`. Despite the method docstring referring to another `RunLog`, an actual `RunLog` operand is rejected by the exact type check in this pinned implementation.

### `__repr__`

Returns a formatted representation of the current state.

```python
@override
__repr__(
    self,
) -> str
```

### `__eq__`

Compares both the current state and inherited operation list.

```python
@override
__eq__(
    self,
    other: object, # Object to compare
) -> bool # Whether both run logs have equal state and operations
```

## Behaviour

`RunLog` instances are unhashable.

In [ ]:
from langchain_core.tracers.log_stream import RunLog, RunLogPatch # Import the real log classes

initial_state = { # Create the initial run state
    "id": "run-1",
    "streamed_output": [],
    "final_output": None,
    "name": "demo_run",
    "type": "chain",
    "logs": {},
}

run_log = RunLog(state=initial_state) # Create the initial RunLog

first_patch = RunLogPatch( # Create the first group of JSON Patch operations
    {
        "op": "add",
        "path": "/streamed_output/-",
        "value": "Hello ",
    },
    {
        "op": "replace",
        "path": "/final_output",
        "value": "Hello ",
    },
)

run_log = run_log + first_patch # Apply the first patch

print("After first patch:") # Display a heading
print(run_log.state) # Display the updated state

second_patch = RunLogPatch( # Create another patch
    {
        "op": "add",
        "path": "/streamed_output/-",
        "value": "Saad!",
    },
    {
        "op": "replace",
        "path": "/final_output",
        "value": "Hello Saad!",
    },
)

final_log = run_log + second_patch # Apply the second patch

print("\nFinal state:") # Display the final state
print(final_log.state)

print("\nStreamed output:") # Display all streamed chunks
print(final_log.state["streamed_output"])

print("\nFinal output:") # Display the aggregated output
print(final_log.state["final_output"])

print("\nAll patch operations:") # Display all accumulated operations
for operation in final_log.ops:
    print(operation)

print("\nRunLog representation:") # Demonstrate __repr__
print(final_log)

equal_log = RunLog( # Create another RunLog with identical data
    *final_log.ops,
    state=final_log.state.copy(),
)

print("\nLogs are equal:", final_log == equal_log) # Demonstrate __eq__

try:
    invalid_result = final_log + equal_log # RunLog can only be added to an exact RunLogPatch
except TypeError as error:
    print("\nInvalid addition:", error) # Display the TypeError

try:
    hash(final_log) # Attempt to hash the RunLog
except TypeError as error:
    print("Hashing error:", error) # Show that RunLog is unhashable

# `LogStreamCallbackHandler: BaseTracer, _StreamingCallbackHandler[Any]`

Tracer that emits run-log updates through an asynchronous in-memory stream.

## Constructor

```python
LogStreamCallbackHandler(
    *,
    auto_close: bool = True, # Whether to close the stream when the root run finishes
    include_names: Sequence[str] | None = None, # Include runs whose names match
    include_types: Sequence[str] | None = None, # Include runs whose types match
    include_tags: Sequence[str] | None = None, # Include runs having a matching tag
    exclude_names: Sequence[str] | None = None, # Exclude runs whose names match
    exclude_types: Sequence[str] | None = None, # Exclude runs whose types match
    exclude_tags: Sequence[str] | None = None, # Exclude runs having a matching tag
    _schema_format: Literal["original", "streaming_events"] = "streaming_events", # Internal schema format
) -> None
```

The schema-format argument is internal and subject to change.

Raises `ValueError` when `_schema_format` is not `"original"` or `"streaming_events"`.

## Methods

### `__aiter__`

Returns the receive stream's asynchronous iterator.

```python
__aiter__(
    self,
) -> AsyncIterator[RunLogPatch] # Iterator yielding emitted patches
```

### `send`

Enqueues one `RunLogPatch` containing the supplied operations.

```python
send(
    self,
    *ops: dict[str, Any], # JSON Patch operations to enqueue
) -> bool # True after the patch is sent
```

The implementation uses `send_nowait()` and returns `True` on the successful path. Stream errors are not caught.

### `tap_output_aiter`

Passes through an asynchronous output iterator while adding each included non-root run chunk to its `streamed_output` log.

```python
async tap_output_aiter(
    self,
    run_id: UUID, # ID of the run producing the output
    output: AsyncIterator[T], # Asynchronous output iterator to tap
) -> AsyncIterator[T] # Original output values
```

A root run is not logged here because its streamed output is handled by the runnable log-stream implementation. Runs excluded from the log are passed through without emitting patches.

### `tap_output_iter`

Passes through a synchronous output iterator while adding each included non-root run chunk to its `streamed_output` log.

```python
tap_output_iter(
    self,
    run_id: UUID, # ID of the run producing the output
    output: Iterator[T], # Synchronous output iterator to tap
) -> Iterator[T] # Original output values
```

A root run is not logged here. Runs excluded from the log are passed through without emitting patches.

### `include_run`

Determines whether a run should appear in the sub-run log.

```python
include_run(
    self,
    run: Run, # Run to evaluate against the configured filters
) -> bool # Whether the run should be included
```

The root run is always excluded from `logs`.

When no inclusion filters are configured, non-root runs are included by default. When one or more inclusion filters are configured, a run is included when its name, type, or at least one tag matches any configured inclusion filter.

Exclusion filters are then applied cumulatively. A matching excluded name or type removes the run, and any tag present in `exclude_tags` removes the run.

## Behaviour

The first created run becomes the root and initializes the streamed `RunState`. Included sub-runs are stored under their names; repeated names receive suffixes such as `":2"` and `":3"`.

Starting an included sub-run emits its ID, name, type, tags, metadata, start time, empty streamed outputs, and unfinished output fields. Under the `"streaming_events"` schema, standardized inputs are also included.

Completing an included sub-run emits its final output and end time. Under the `"streaming_events"` schema, its inputs are replaced with their finalized value.

Streaming model tokens add the token to `streamed_output_str`. `streamed_output` receives the underlying message for a `ChatGenerationChunk`; otherwise it receives the token value.

When `auto_close=True`, completing the root run closes the send stream.

In [ ]:
import asyncio # Import asyncio for running the pipeline and log consumer together

from langchain_core.runnables import RunnableLambda # Import the real LangChain runnable
from langchain_core.tracers.log_stream import LogStreamCallbackHandler, RunLog, RunLogPatch # Import the real log-stream classes


def double(number: int) -> int: # Define the first pipeline step
    return number * 2 # Double the input number


def format_result(number: int) -> str: # Define the second pipeline step
    return f"Final value: {number}" # Convert the number into formatted text


double_step = RunnableLambda(double).with_config( # Create the first named runnable
    run_name="double"
)

format_step = RunnableLambda(format_result).with_config( # Create the second named runnable
    run_name="format_result"
)

pipeline = double_step | format_step # Combine both runnables into one pipeline

handler = LogStreamCallbackHandler( # Create the real log-stream handler
    auto_close=True, # Close the stream when the root pipeline finishes
    include_names=["double", "format_result"], # Include only these child runs
)


async def run_pipeline() -> None: # Define the asynchronous demonstration
    task = asyncio.create_task( # Start the pipeline in a separate task
        pipeline.ainvoke( # Invoke the real runnable pipeline
            5, # Provide the pipeline input
            config={ # Configure the root run
                "callbacks": [handler], # Attach the real handler
                "run_name": "number_pipeline", # Name the root run
                "tags": ["demo"], # Add a run tag
                "metadata": {"source": "jupyter"}, # Add run metadata
            },
        )
    )

    combined_log: RunLogPatch | RunLog | None = None # Store the reconstructed log

    async for patch in handler: # Receive actual patches emitted by the handler
        print("\nPatch operations:") # Display a patch heading

        for operation in patch.ops: # Visit each JSON Patch operation
            print(operation) # Display the operation

        if combined_log is None: # Check whether this is the first patch
            combined_log = patch # Store the initial state patch
        else:
            combined_log = combined_log + patch # Apply the patch to the current log

    result = await task # Retrieve the pipeline result

    print("\nPipeline result:", result) # Display the final runnable output

    if isinstance(combined_log, RunLog): # Check that a complete log was reconstructed
        print("\nFinal reconstructed state:") # Display a heading
        print(combined_log.state) # Display the complete run state

        print("\nLogged child runs:") # Display a child-run heading

        for log_name, entry in combined_log.state["logs"].items(): # Visit child logs
            print("Name:", log_name) # Display the generated log name
            print("Type:", entry["type"]) # Display the run type
            print("Input:", entry.get("inputs")) # Display the child input
            print("Output:", entry["final_output"]) # Display the child output


await run_pipeline() # Run directly inside Jupyter Notebook